# 1. Classical: find the exact answers

Run this notebook, then [quantum](quantum_capacity_constrained_clustering.ipynb),
then [results](results.ipynb), using the `.venv` kernel.
We run **all supplied demand vectors**: currently ten S instances, each
with 4 customers, 2 vehicles and capacity 7. A detailed `s-0` example follows.
Algorithms stay inline; shared helpers handle inputs, files and QUBO checks.

## The clustering problem (A1)

Group nearby customers into vehicles. Every customer is served once;
each vehicle's total demand is at most $Q$. Unused vehicles are allowed.

`labels = (0, 0, 1, 1)` assigns customers 0 and 1 to vehicle 0,
and customers 2 and 3 to vehicle 1. Labels are **not** visit orders.

With $x_{iv}=1$ when vehicle $v$ serves customer $i$, and pair cost $c_{ij}$:
$$F=\sum_v\sum_{i<j}c_{ij}x_{iv}x_{jv},\qquad
\sum_v x_{iv}=1,\qquad \sum_i d_i x_{iv}\le Q.$$

Minimize the sum of costs within groups, subject to the two rules.
Customer coordinates stay in the input to define $c_{ij}$; we only need
the resulting pair costs for this walkthrough.

In [1]:
from pathlib import Path
from collections import Counter
from dataclasses import asdict
from itertools import combinations, product
from time import perf_counter
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "track_a" / "inputs" / "seeds_s.json").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from track_a.common.instances import INPUT_PATH, OUTPUT_PATH, Instance, ensure_instances, feasible
from track_a.common.analysis import fingerprint, save
from track_a.common.models import ENCODING, Qubo, build_qubo, encode, validate_small_qubo
from track_a.common.classical import installed_qubo_crosscheck

instances = ensure_instances()
output = OUTPUT_PATH
shots, seed = 256, 0
display(pd.DataFrame([
    {"instance": i.id, "demands": i.demands, "certificate": i.certificate, **i.ratios}
    for i in instances
]))

,instance,demands,certificate,rho,largest_demand_ratio
0,s-0,"(2, 3, 4, 2)","(0, 0, 1, 0)",0.785714,0.571429
1,s-1,"(3, 4, 2, 2)","(0, 0, 1, 1)",0.785714,0.571429
2,s-2,"(1, 2, 3, 4)","(0, 0, 0, 1)",0.714286,0.571429
3,s-3,"(2, 2, 4, 3)","(0, 0, 1, 0)",0.785714,0.571429
4,s-4,"(4, 2, 3, 2)","(0, 0, 1, 1)",0.785714,0.571429
5,s-5,"(2, 4, 1, 3)","(0, 0, 0, 1)",0.714286,0.571429
6,s-6,"(3, 2, 4, 2)","(0, 0, 1, 0)",0.785714,0.571429
7,s-7,"(4, 2, 2, 3)","(0, 0, 1, 1)",0.785714,0.571429
8,s-8,"(2, 1, 3, 2)","(0, 0, 0, 1)",0.571429,0.428571
9,s-9,"(2, 3, 2, 4)","(0, 0, 0, 1)",0.785714,0.571429


## 1. Score a grouping

Add distances only between customers sharing a vehicle.
`canonical` renames vehicles in order of first appearance:
`(1, 1, 0, 0)` becomes `(0, 0, 1, 1)`. This avoids counting the same
grouping twice. The input certificate proves feasibility, not optimality.

In [2]:
def objective(instance: Instance, labels: tuple[int, ...] | list[int]) -> float:
    distances = instance.distances
    return float(sum(distances[i, j] for i in range(instance.n)
                     for j in range(i) if labels[i] == labels[j]))


def canonical(labels: tuple[int, ...] | list[int]) -> tuple[int, ...]:
    mapping: dict[int, int] = {}
    return tuple(mapping.setdefault(k, len(mapping)) for k in labels)

## 2. List the possible groupings

Four customers and two vehicles give only $2^4=16$ labeled assignments.
Keep one representative per vehicle relabeling; filter capacity later.
This direct search is intended for the current S-only scope.

In [3]:
def partitions(n: int, vehicles: int):
    for labels in product(range(vehicles), repeat=n):
        if labels == canonical(labels):
            yield labels

## 3. Find the exact clustering optimum

Score every feasible grouping and keep the minimum. Count tied optima
within $10^{-8}$, without double-counting vehicle relabelings.
Independently solve the target QUBO with CPLEX and check its decoded
grouping has the same clustering cost. There is no routing in this search.

In [4]:
def clustering_reference(instance: Instance) -> dict:
    """Exact clustering only; routing references belong to the results stage."""
    start = perf_counter()
    candidates = [labels for labels in partitions(instance.n, instance.vehicles)
                  if feasible(instance, labels)]
    if not candidates:
        raise ValueError("No feasible grouping")
    costs = [objective(instance, labels) for labels in candidates]
    best = min(costs)
    result = {
        "clustering": {"cost": best, "labels": candidates[costs.index(best)],
                       "distinct_optima": sum(abs(cost - best) <= 1e-8 for cost in costs),
                       "status": "proven_optimum"},
        "status": "proven_optimum", "algorithm": "exhaustive-clustering-partitions",
        "settings": {"maximum_customers": 4, "vehicle_labels_quotiented": True,
                     "equality_tolerance": 1e-8},
        "feasible_partitions": len(candidates), "seconds": perf_counter() - start,
    }
    result["cplex_crosscheck"] = {
        "clustering": installed_qubo_crosscheck(instance, build_qubo(instance), best),
    }
    return result

## 4. Sample the QUBO with classical annealing

Start each shot from random bits. Try one bit flip per step; accept
improvements, and sometimes uphill moves with probability $e^{-\Delta E/T}$.
Cool geometrically from `10 * penalty` to `0.01 * penalty` over 400 steps.
Chains run together for efficiency. No warm start or repair.
The [quantum notebook](quantum_capacity_constrained_clustering.ipynb) derives the QUBO.

In [5]:
def anneal(qubo: Qubo, shots: int, seed: int, steps: int = 400) -> Counter[str]:
    """Independent Metropolis chains, no repair or certificate warm start."""
    rng = np.random.default_rng(seed)
    size = len(qubo.variable_order)
    z = rng.integers(0, 2, size=(shots, size))
    diagonal = np.diag(qubo.matrix)
    pairs = 2 * (qubo.matrix - np.diag(diagonal))
    fields = z @ pairs + diagonal
    temperatures = np.geomspace(10 * qubo.penalty, 0.01 * qubo.penalty, steps)
    rows = np.arange(shots)
    for temperature in temperatures:
        columns = rng.integers(size, size=shots)
        change = 1 - 2 * z[rows, columns]
        delta = change * fields[rows, columns]
        accepted = (delta <= 0) | (np.log(rng.random(shots)) < -delta / temperature)
        selected_rows, selected_cols = rows[accepted], columns[accepted]
        fields[selected_rows] += change[accepted, None] * pairs[selected_cols]
        z[selected_rows, selected_cols] = 1 - z[selected_rows, selected_cols]
    return Counter("".join(map(str, row)) for row in z)

## 5. Run the three classical comparisons

Save exact references, exhaustive QUBO checks, configurations and raw counts.
Compare exact constrained search, QUBO annealing and uniform random bits.
All use 256 shots and seed 0. The exact solver's one answer is repeated
256 times; those are **not independent samples**.

In [6]:
def classical_runs(instances, output: Path, shots: int, seed: int) -> None:
    for instance in instances:
        print(f"Classical {instance.id}", flush=True)
        record = asdict(instance)
        reference = clustering_reference(instance)
        reference["instance_hash"] = fingerprint(record)
        save(output / "references" / f"{instance.id}.json", reference)
        qubo = build_qubo(instance)
        if instance.tier == "s":
            validation = validate_small_qubo(instance, reference["clustering"]["cost"])
            validation["installed_qubo_solver"] = reference["cplex_crosscheck"]["clustering"]
            save(output / "validation" / f"{instance.id}.json",
                 validation)
        config = {"shots": shots, "seed": seed, "annealing_steps": 400,
                  "initial_temperature": 10 * qubo.penalty,
                  "final_temperature": 0.01 * qubo.penalty,
                  "cooling": "geometric", "restarts": shots,
                  "warm_start": False, "repair": False, "penalty": qubo.penalty,
                  "time_limit_seconds": None}
        for method in ("classical-constrained", "classical-qubo-sa", "random-bits"):
            start = perf_counter()
            if method == "classical-constrained":
                labels = min(
                    (labels for labels in partitions(instance.n, instance.vehicles)
                     if feasible(instance, labels)),
                    key=lambda labels: objective(instance, labels),
                )
                counts = {encode(instance, labels): shots}
                seconds = perf_counter() - start
            elif method == "classical-qubo-sa":
                counts = dict(anneal(qubo, shots, seed))
                seconds = perf_counter() - start
            else:
                rng = np.random.default_rng(seed)
                bits = rng.integers(0, 2, (shots, len(qubo.variable_order)))
                counts = dict(Counter("".join(map(str, row)) for row in bits))
                seconds = perf_counter() - start
            save(output / "runs" / f"{instance.id}-{method}-{seed}.json", {
                "instance": record, "instance_id": instance.id, "instance_hash": fingerprint(record),
                "method": method, "seed": seed, "encoding": ENCODING,
                "configuration_id": fingerprint({**config, "method": method}),
                "configuration": {**config, "method": method},
                "status": "completed", "counts": counts, "shot_budget": shots,
                "atom_count": len(qubo.variable_order), "clustering_seconds": seconds,
                "deterministic_solver_result_repeated": method == "classical-constrained",
                "variable_order": qubo.variable_order,
            })

In [7]:
instance_path = output / "instances.json"
if instance_path.exists():
    previous = json.loads(instance_path.read_text())
    current = [{**asdict(i), **i.ratios, "certificate_valid": True} for i in instances]
    if fingerprint(previous) != fingerprint(current):
        raise ValueError("Use a new output directory for changed input data")
save(instance_path, [{**asdict(i), **i.ratios, "certificate_valid": True} for i in instances])
save(output / "generation.json", {
    "input_path": str(INPUT_PATH.resolve()), "numpy": np.__version__,
    "coordinates": "explicit input coordinates, shared across demand vectors",
    "selected_demand_indices": [i.instance_seed for i in instances],
    "input": json.loads(INPUT_PATH.read_text()),
})
classical_runs(instances, output, shots, seed)
references = {i.id: json.loads((output / "references" / f"{i.id}.json").read_text())
              for i in instances}
display(pd.DataFrame([
    {"instance": name, "model_reference": ref["clustering"]["cost"],
     "status": ref["status"],
     "model_optima": ref["clustering"]["distinct_optima"]}
    for name, ref in references.items()
]))

Classical s-0


Classical s-1


Classical s-2


Classical s-3


Classical s-4


...

Classical s-6


Classical s-7


Classical s-8


Classical s-9


,instance,model_reference,status,model_optima
0,s-0,80.886853,proven_optimum,1
1,s-1,80.886853,proven_optimum,1
2,s-2,80.886853,proven_optimum,1
3,s-3,80.886853,proven_optimum,1
4,s-4,80.886853,proven_optimum,1
5,s-5,80.886853,proven_optimum,1
6,s-6,80.886853,proven_optimum,1
7,s-7,80.886853,proven_optimum,1
8,s-8,80.886853,proven_optimum,1
9,s-9,80.886853,proven_optimum,1


## 6. Check one clustering by hand: `s-0`

The benchmark above runs all vectors. Here we work through only `s-0`:
demands **[2, 3, 4, 2]**, two vehicles, capacity **7**.
Customer numbers start at **0**.

1. Use the six pair costs in the table below.
2. Check all eight distinct groupings. Sum each group's demands; reject it
   if either load exceeds 7. Exactly four groupings are feasible here.
3. For each feasible grouping, sum costs of pairs in the **same** group.
   Pick the smallest total.

The winning groups are **{0, 3} and {1, 2}**:
- Loads: $2+2=4$ and $3+4=7$; both fit.
- Pair costs: $c_{03}\approx9.570242$ and $c_{12}\approx71.316611$.
- Clustering cost: $9.570242+71.316611\approx\mathbf{80.886853}$.

This completely checks the clustering answer; no visiting order or depot
tour is needed. The tables recalculate if you edit the input, while the
worked numbers above describe the shipped input.

In [8]:
manual_instance = instances[0]
manual_reference = references[manual_instance.id]
display(pd.DataFrame({"customer": range(manual_instance.n), "demand": manual_instance.demands}))
display(pd.DataFrame([
    {"pair": (i, j), "cost": manual_instance.distances[i, j]}
    for i, j in combinations(range(manual_instance.n), 2)
]).round(6))

manual_candidates = []
for labels in partitions(manual_instance.n, manual_instance.vehicles):
    groups = [tuple(i for i, v in enumerate(labels) if v == vehicle)
              for vehicle in range(manual_instance.vehicles)]
    manual_candidates.append({
        "labels": labels, "groups": groups,
        "loads": [sum(manual_instance.demands[i] for i in group) for group in groups],
        "feasible": feasible(manual_instance, labels),
        "within_group_cost": objective(manual_instance, labels),
    })
display(pd.DataFrame(manual_candidates).round(6))
manual_labels = tuple(manual_reference["clustering"]["labels"])
print("Best clustering:", manual_labels, "| cost:", objective(manual_instance, manual_labels))

,customer,demand
0,0,2
1,1,3
2,2,4
3,3,2


,pair,cost
0,"(0, 1)",46.262020
1,"(0, 2)",46.548088
2,"(0, 3)",9.570242
3,"(1, 2)",71.316611
4,"(1, 3)",51.722215
5,"(2, 3)",37.671375


,labels,groups,loads,feasible,within_group_cost
0,"(0, 0, 0, 0)","[(0, 1, 2, 3), ()]","[11, 0]",False,263.090550
1,"(0, 0, 0, 1)","[(0, 1, 2), (3,)]","[9, 2]",False,164.126719
2,"(0, 0, 1, 0)","[(0, 1, 3), (2,)]","[7, 4]",True,107.554477
3,"(0, 0, 1, 1)","[(0, 1), (2, 3)]","[5, 6]",True,83.933395
4,"(0, 1, 0, 0)","[(0, 2, 3), (1,)]","[8, 3]",False,93.789704
5,"(0, 1, 0, 1)","[(0, 2), (1, 3)]","[6, 5]",True,98.270303
6,"(0, 1, 1, 0)","[(0, 3), (1, 2)]","[4, 7]",True,80.886853
7,"(0, 1, 1, 1)","[(0,), (1, 2, 3)]","[2, 9]",False,160.710201


Best clustering: (0, 1, 1, 0) | cost: 80.88685325702023


Next: [quantum clustering](quantum_capacity_constrained_clustering.ipynb).
The required CVRP reference and fixed-group route evaluation appear only in
[results](results.ipynb), not in the clustering optimization.

All supplied demand vectors are run. Inputs include validated feasibility
witnesses, but are explicit rows rather than Track A's seed-based generator (A2).